# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Overview of Croissant entities using their `@id` fields.

print("Available record sets and their @id's:")
for record_set in metadata.record_sets:
    print(f"- @id: {record_set['@id']} | name: {record_set.get('name', '')}")

print("\nFields within each record set:")
for record_set in metadata.record_sets:
    print(f"\nRecord set @id: {record_set['@id']} ({record_set.get('name','')})")
    fields = record_set.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]  # If only one field, wrap in list
    for field in fields:
        print(f"  - Field @id: {field['@id'] if isinstance(field, dict) and '@id' in field else field}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Select record set(s) for extraction using @id
record_set_ids = [rec['@id'] for rec in metadata.record_sets]
print(f"Extracting these record sets: {record_set_ids}")

dataframes = {}
for record_set_id in record_set_ids:
    print(f"\nLoading records for record_set '@id': {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for {record_set_id}, columns: {df.columns.tolist()}")

# Take the first record set ID for step 4 EDA, if available
primary_record_set_id = record_set_ids[0] if record_set_ids else None
if primary_record_set_id:
    print(f"\nPreview of DataFrame for {primary_record_set_id}:")
    display(dataframes[primary_record_set_id].head())
else:
    print("No record set found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np

# We'll use the first record set as an example (change this if you desire)
record_set_id = primary_record_set_id
df = dataframes[record_set_id]
print(f"Number of records: {len(df)}")

# Display column names with their Croissant field @id if available
print("\nColumns in this DataFrame:")
for col in df.columns:
    print(f"- {col}")

# Select a numeric field for analysis
# We'll attempt to automatically detect numeric columns. Adjust as desired.
numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
if not numeric_fields:
    # Try to convert plausible columns to numeric
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
        except Exception:
            continue
    numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()

if numeric_fields:
    numeric_field = numeric_fields[0]
    print(f"\nUsing numeric field for analysis: {numeric_field}")
else:
    print("No numeric field found in DataFrame.")
    numeric_field = None

if numeric_field is not None:
    threshold = df[numeric_field].mean() if pd.notnull(df[numeric_field].mean()) else 0
    filtered_df = df[df[numeric_field] > threshold]
    print(f"\nFiltered records with {numeric_field} > mean ({threshold:.2f}):")
    print(filtered_df.head())

    # Normalization
    col_norm = f"{numeric_field}_normalized"
    filtered_df[col_norm] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, col_norm]].head())

    # Try grouping by a non-numeric field if available
    group_fields = [col for col in df.columns if col != numeric_field and not pd.api.types.is_numeric_dtype(df[col])]
    group_field = group_fields[0] if group_fields else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped data by {group_field} (mean of {numeric_field}):")
        print(grouped_df.head())
else:
    print("Skipping EDA steps (no numeric field found).")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

if numeric_field is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If there is a group_field, visualize mean by group
    if group_field:
        plt.figure(figsize=(8,4))
        mean_vals = df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False)
        sns.barplot(x=mean_vals.index.astype(str), y=mean_vals.values)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field to visualize.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

**Key observations:**

* The dataset was successfully loaded using its Croissant metadata schema.
* The available record sets, fields, and columns were enumerated by their `@id` for reference.
* Data was explored and filtered using a detected numeric field, with normalization and grouped analysis illustrated as examples.
* Data distributions and relationships can be visualized for deeper investigation.

Further analysis can be tailored by referencing additional field or record set `@id`s using the above overview and the `mlcroissant` library's methods.